### Suchen auf Graphen, Tiefen- und Breitensuche
(auch genannt **DFS** (depth-first search) und **BFS** (breadth-first search).

Probleme die DFS löst:
- Weg durch Labyrinth finden,
- Sudoku lösen,
- Zusammenhangskomponente finden.

Probleme die BFS löst:
- kürzester Weg in Graph/Netzwerk finden,
- (kleine) Sokoban Probleme lösen.

Wir erläutern diese Suchstrategien anhand folgender Situation.
Ein Höhlenforscher hat die Aufgabe, ein grosses unterirdisches Höhlensystem zumindest zum Teil zu kartieren.
Das System besteht aus vielen Kammern,
welche durch kurze Tunnels verbunden sind. 
2 Kammern sind durch höchstens einen Tunnel verbunden. 
Jede Kammer ist mit einer eindeutigen Zahl markiert, und jeder Tunneleingang zeigt die Nummer der
Kammer zu der dieser führt (die Tunnel sind eher Türen mit Zimmernummern).    
Der Forscher ist in einer ersten Kammer, von wo aus er seine Erkundungen beginnt.
Er darf keine Markierungen am Höhlensystem anbringen, darf sich aber Notizen machen, will diese aber knapp halten.
Er verwaltet 

1. eine Liste  `zu_besuchende_Kammern` mit noch unbesuchten Kammern,
2. einen Dictionary `go_back`. Der Wert ist die Kammer(nummer), die der Tunnel verlässt, der
  Schlüssel ist die Kammer(nummer), wo der Tunnel hinführt.


Das allgemeine Vorgehen ist Folgendes:

- eine Liste  `zu_besuchende_Kammern` mit noch unbesuchten Kammern,
- ein Dictionary `go_back`: Der Wert ist die Kammer(nummer), die der Tunnel verlässt, der
  Schlüssel ist die Kammer(nummer), wo der Tunnel hinführt.



   
- Gehe nacheinander durch alle ausgehenden Tunnels zur nächsten Kammer.  
  Markiere den Tunnelausgang mit der Nummer der Kammer von welcher du kommst (**damit man wieder zurück findet**).  
  Kehre zur 
- markiere neue Kammer mit eindeutiger Laufnummer,
- markiere benutzen  Eingang Mit Nummer der letzten Kammer
  nummeriere die verbleibenden Ausgaenge mit 1?, 2? ,,,,
  (ueberschreibe dann 2? auf dem Rueckweg mit der Kammernummer zu dem dieses Tor fuehrt)
 



Strategien:
- depth first:
  besuche immer erste neue Kammer gehe durch Tuer 1? falls diese Tuer zu neuer Kammer fuehrt (sonst komme zurueck und
  probiere naeachte Tuere.
  oder: gesuche alle, damit wir 2? ueberschreiben keonnen. (- gleich wie markierter Eingang!)
  
  kehre um, wenn es nicht mehr weiter geht (bez. keine neue Kammern)

- breadth first:
  arbeitet dich konzentrisch vor,
  alle Kammern mit Abstand 1 vom Start
  alle Kammern mit Abstand 2 vom Start
  Abstand gemaess Notizbuch: darf nur Ausgaenge benutzen mit Kammernummer.


- best guess:
  suchen z.B. Ausgang: messen Luftstrom, je groesser, desto naeher beim ausgang.
  erforsche von "best option" aus
  
  
  
Zusammenhangskomponente:

Kammern und Zimmernummern tragen Farben. Koennen nur Turen einr Fare oeffnen.


  
  


&uuml; &ouml; &auml;

In [199]:
def get_items(items):
    while items:
        yield items.pop()

In [206]:
xs = [1, 2, 3]
list(get_items(xs)), xs

([3, 2, 1], [])

In [212]:
ys = [3, 4, 5]
numbers = get_items(ys)

In [216]:
print(next(numbers))
ys.extend([6, 7])
print(next(numbers))
ys

6
7


[3, 4, 6]

In [148]:
from collections import deque


def path_home(node, go_back):
    path = []
    while node:
        path.append(node)
        node = go_back[node]

    return path[::-1]


def search_df(node, get_options, is_goal):
    '''node: start node'''
    stack = [node]
    go_back = {node: None}

    while stack and (node := stack.pop()) and not is_goal(node):
        for opt in get_options(node):
            if opt in go_back:
                continue
            go_back[opt] = node
            stack.append(opt)

    return node, go_back


def search_bf(node, get_options, is_goal):
    '''node: start node'''
    queue = deque([node])
    go_back = {node: None}

    while queue and (node := queue.pop()) and not is_goal(node):
        for opt in get_options(node):
            if opt in go_back:
                continue
            go_back[opt] = node
            queue.appendleft(opt)

    return node, go_back

In [ ]:
# consider using an exclude in get_options
def search_df(node, get_options, is_goal):
    '''node: start node'''
    stack = [node]
    go_back = {node: None}

    while stack and (node := stack.pop()) and not is_goal(node):
        for opt in get_options(node, exlude=go_back):
            go_back[opt] = node
            stack.append(opt)

    return node, go_back

In [ ]:
def search_dfg(node, get_options, is_goal):
    '''node: start node'''
    stack = []
    go_back = {node: None}

    while not is_goal(node):
        succs = get_options(node)
        if (succ := next(succs, None)):
            stack.append(succs)
        if succ in go_back:
            continue
        go_back[succ] = node
        stack.append(succs)


    return succ, go_back

In [189]:
def get_options(queens, n_queens=8, exclude=()):
    col = len(queens)
    if col == n_queens:
        return

    for row in range(n_queens):
        if is_attacked(queens, (col, row)):
            continue
        queens_ = queens + (row,)
        if queens_ not in exclude:
            yield queens_

In [184]:
def search_dfg(node, get_options, is_goal):
    '''node: start node'''
    go_back = {node: None}
    stack = [(node, get_options(node, exclude=go_back))]

    while stack:
        node, succs = stack[-1]
        if is_goal(node):
            break

        if (succ := next(succs, None)):
            go_back[succ] = node
            stack.append((succ, get_options(succ, exclude=go_back)))
        else:
            stack.pop()


    return node, go_back


# may let get_options depend on the stack and child_parent
def search_dfg(node, get_options, is_goal):
    '''node: start node'''
    go_back = {node: None}
    stack = [node]

    children = get_children(stack, go_back)

    while len(go_back) < dims[0]*dims[1]:
        if (child := next(children, None)):
            go_back[child] = node
            stack.append(child)
        else:
            stack.pop()


    return go_back


In [191]:
goal, go_back = search_dfg((6,), get_options, is_goal)
goal

(6, 0, 2, 7, 5, 3, 1, 4)

In [192]:
def search_dfg(node, get_options, is_goal):
    '''node: start node'''
    go_back = {node: None}
    stack = [(node, get_options(node, exclude=go_back))]

    while stack and not (node := is_goal(stack[-1][0])):
        succs = stack[-1][1]
        if (succ := next(succs, None)):
            go_back[succ] = node
            stack.append((succ, get_options(succ, exclude=go_back)))
            node = succ
        else:
            stack.pop()

    return node, go_back

In [193]:
goal, go_back = search_dfg((6,), get_options, is_goal)
goal

(6, 0, 2, 7, 5, 3, 1, 4)

In [ ]:
def search_dfg(node, get_options, is_goal):
    '''node: start node'''
    go_back = {node: None}
    stack = [(node, get_options(node, exclude=go_back))]

    while stack and not (node := is_goal(stack[-1][0])):
        succs = stack[-1][1]
        if (succ := next(succs, None)):
            go_back[succ] = node
            stack.append((succ, get_options(succ, exclude=go_back)))
            node = succ
        else:
            stack.pop()

    return node, go_back

In [ ]:
stack = [(0, 1)]
while stack and 

In [165]:
def is_connected(p, q):
    '''p, q: Positions on a chessboard
       returns True if a queen at p attackes a queen at q
    '''
    return (p[0] == q[0]
            or p[1] == q[1]
            or abs(p[0] - q[0]) == abs(p[1] - q[1])
            )


def is_attacked(queens, p):
    for col, row in enumerate(queens):
        if is_connected((col, row), p):
            return True


def get_options(queens, n_queens=8):
    col = len(queens)
    if col == n_queens:
        return

    for row in reversed(range(n_queens)):
        if not is_attacked(queens, (col, row)):
            yield queens + (row,)


def is_goal(queens):
    return len(queens) == 8

In [166]:
goal, go_back = search_df((0,), get_options, is_goal)
goal

(0, 4, 7, 5, 2, 6, 1, 3)

In [167]:
goal, go_back = search_bf((0,), get_options, is_goal)
goal

(0, 6, 4, 7, 1, 3, 5, 2)

In [168]:
goal, go_back = search_dfg((0,), get_options, is_goal)
goal

(0,)

In [139]:
def is_goal(queens):
    return len(queens) == 8


def get_succs(queens, n_queens=8):
    col = len(queens)
    if col == n_queens:
        return

    for row in range(n_queens):
        if not is_attacked(queens, (col, row)):
            yield queens + (row,)


def search_bt(node):
    if is_goal(node):
        return node

    for next_node in get_succs(node):
        if (goal := search_bt(next_node)):
            return goal

In [141]:
search_bt(())

(0, 4, 7, 5, 2, 6, 1, 3)

Liste mit schwierigen Sudokus:  
[Schwierige Sudokus](https://github.com/nurdymuny/sudoky-energy/blob/master/puzzles/hardest/hardest_puzzles.txt)|

In [97]:
s = '''\
1....527.
.5....8..
.4.92..53
...6.298.
.2..91...
79....6.2
5.41.....
...267541
...5.....
'''
print(s)

1....527.
.5....8..
.4.92..53
...6.298.
.2..91...
79....6.2
5.41.....
...267541
...5.....



In [98]:
def sudoku2str(sudoku):
    s = '\n'.join([''.join(str(n) for n in sudoku[9*i:9*(i+1)]) for i in range(9)])
    return s.replace('0', '.')


def str2sudoku(s):
    s = s.replace('\n', '').replace('.', '0')
    return [int(x) for x in s]

In [99]:
sudoku = str2sudoku(s)
sudoku[:3], len(sudoku)

([1, 0, 0], 81)

In [ ]:
print(sudoku2str(sudoku))

In [ ]:
def get_row(sudoku, row):
    return (n for i in range(9)
            if (n := sudoku[9*row+i]) > 0)


def get_col(sudoku, col):
    return (n for j in range(9)
            if (n := sudoku[9*j + col]) > 0)


def get_block(sudoku, blk):
    row = (blk // 3) * 3
    col = (blk % 3) * 3
    return (n for i in range(3) for j in range(3)
            if (n := sudoku[9*(row + i) + col + j]) > 0)


def are_unique(items):
    seen = set()
    for item in items:
        if item in seen:
            return False
        seen.add(item)
    return True


def is_sudoku(sudoku):
    for get_group in (get_row, get_col, get_block):
        for i in range(9):
            if not are_unique(get_group(sudoku, i)):
                return False
    return True

In [85]:
is_sudoku(sudoku)

True

In [ ]:
list(get_block(sudoku, 8))

In [94]:
def is_goal(sudoku):
    return 0 not in sudoku


def get_options(sudoku, i):
    row, col = divmod(i, 9)
    block = 3*(row//3) + col//3

    for opt in range(1, 10):
        if (opt in get_row(sudoku, row)
                or opt in get_col(sudoku, col)
                or opt in get_block(sudoku, block)):
            continue
        yield opt
        # sudoku[i] = opt
        # yield tuple(sudoku)


def get_options(sudoku):
    i = sudoku.index(0)
    row, col = divmod(i, 9)
    block = 3*(row//3) + col//3

    for opt in range(1, 10):
        if (opt in get_row(sudoku, row)
                or opt in get_col(sudoku, col)
                or opt in get_block(sudoku, block)):
            continue

        sudoku[i] = opt
        yield tuple(sudoku)

In [196]:
[1,2,2,2].index(2, 2)

2

In [ ]:
search(tuple(sudoku), get_options, is_goal)

In [91]:
def solve(sudoku):
    if 0 not in sudoku:
        return True

    i = sudoku.index(0)  # erstes leeres Feld
    for opt in get_options(sudoku, i):
        sudoku[i] = opt
        if solve(sudoku):
            return True
        sudoku[i] = 0

In [109]:
s = '''\
1....527.
.5....8..
.4.92..53
...6.298.
.2..91...
79....6.2
5.41.....
...267541
...5.....
'''
my_sudoku = str2sudoku(s)


print(sudoku2str(my_sudoku))
print()
solve(my_sudoku)
print(sudoku2str(my_sudoku))

1....527.
.5....8..
.4.92..53
...6.298.
.2..91...
79....6.2
5.41.....
...267541
...5.....

136485279
952713864
847926153
413672985
625891437
798354612
564139728
389267541
271548396


In [113]:
print(sudoku2str(my_sudoku))
print()
solve(my_sudoku)
print(sudoku2str(my_sudoku))

1....7.9.
.3..2...8
..96..5..
..53..9..
.1..8...2
6....4...
3......1.
.4......7
..7...3..

162857493
534129678
789643521
475312986
913586742
628794135
356478219
241935867
897261354


In [102]:
print(sudoku2str(sudoku))

136485279
952713864
847926153
413672985
625891437
798354612
564139728
389267541
271548396


In [100]:
set(get_options(sudoku, 1))

{3, 6, 8}

In [101]:
solve(sudoku)

True

In [107]:
escargot = str2sudoku('..3.8.9...4.3....6.2.7.1...8...69.4.5....9.2...6...47...2...8.1.9..6..7...1..3.4.6.')
print(sudoku2str(escargot))
print()
solve(escargot)
print(sudoku2str(escargot))

..3.8.9..
.4.3....6
.2.7.1...
8...69.4.
5....9.2.
..6...47.
..2...8.1
.9..6..7.
..1..3.4.

..3.8.9..
.4.3....6
.2.7.1...
8...69.4.
5....9.2.
..6...47.
..2...8.1
.9..6..7.
..1..3.4.


In [110]:
is_sudoku(escargot)

False

In [111]:
s = '100007090030020008009600500005300900010080002600004000300000010040000007007000300'
print(sudoku2str(s))

1....7.9.
.3..2...8
..96..5..
..53..9..
.1..8...2
6....4...
3......1.
.4......7
..7...3..


In [112]:
my_sudoku = str2sudoku(s)
is_sudoku(my_sudoku)

True